### Installation

In [1]:
#!pip install -q unsloth
# >>> NOW: Runtime → Restart session, then run Cell 2 onward (do NOT re-run this cell) <<<

### Cell 2 — load the model (disable torch.compile FIRST)
Colab's torch/Inductor are mismatched and crash at `trainer.train()`; LoRA doesn't need
compile (runs eager). These env vars MUST be set before importing unsloth.

In [2]:
import os
os.environ["TORCHDYNAMO_DISABLE"]     = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

from unsloth import FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,   # match the 30B's raw extraction; language only
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none", random_state = 42,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

### Cell 3 — auth + load + convert the dataset
Auth via Colab Secrets (no interactive widget). Our examples are `{image, messages}`
where the user message already holds the prompt text + an image placeholder; reshape to
Unsloth conversations with the image inlined.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token = userdata.get("HF_TOKEN"))            # uses the Colab Secret; no popup

from datasets import load_dataset
raw   = load_dataset("MitchMooney/para-vlm-extraction", split = "train")
proof = raw                     # PROOF run; use `proof = raw` for the FULL model

def text_of(messages, role):
    return next(c["text"] for m in messages if m["role"] == role
               for c in m["content"] if c["type"] == "text")

def to_conv(ex):
    return {"messages": [
        {"role": "user", "content": [
            {"type": "text",  "text": text_of(ex["messages"], "user")},
            {"type": "image", "image": ex["image"]},
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": text_of(ex["messages"], "assistant")},
        ]},
    ]}

train_conv = [to_conv(ex) for ex in proof]
print(len(train_conv), "examples | sample target:",
      train_conv[0]["messages"][1]["content"][0]["text"][:100])

### Cell 4 — train
Proof = `max_steps=30` (~5 min). For the FULL model: in Cell 3 set `proof = raw`, then
here use the FULL values shown in comments.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_conv,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,    # FULL: 8
        #max_steps = 30,                     # FULL: delete this line, add  num_train_epochs = 1
        num_train_epochs = 1,
        learning_rate = 2e-4,
        #warmup_steps = 5,                   # FULL: delete this, add  warmup_ratio = 0.03
        warmup_ratio = 0.03
        logging_steps = 5,                  # FULL: 5
        save_steps = 100, save_total_limit = 2,
        optim = "adamw_8bit", weight_decay = 0.01,
        output_dir = "outputs", report_to = "none",
        remove_unused_columns = False, dataset_kwargs = {"skip_prepare_dataset": True},
        max_seq_length = 2048,              # Qwen3-VL-8B's max in Unsloth
    ),
)
trainer.train()                             # watch the loss fall (~0.1 by the end)

### Cell 5 — sanity-check on a held-out page
The image goes through the PROCESSOR (builds pixel_values) — NOT as `generate(images=...)`,
which raises "model_kwargs not used: ['images']".

In [ ]:
FastVisionModel.for_inference(model)
ho = load_dataset("MitchMooney/para-vlm-extraction", split = "holdout")[0]
prompt = text_of(ho["messages"], "user")
msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]

input_text = tokenizer.apply_chat_template(msgs, add_generation_prompt = True)
inputs = tokenizer(ho["image"], input_text, add_special_tokens = False, return_tensors = "pt").to("cuda")

from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 1024,
                   streamer = TextStreamer(tokenizer, skip_prompt = True))
print("\n--- GOLD ---\n", text_of(ho["messages"], "assistant"))   # compare

### Cell 6 — save the adapter

In [ ]:
model.save_pretrained("qwen3vl-para-lora")
tokenizer.save_pretrained("qwen3vl-para-lora")
# push to the Hub (uses the same Colab Secret):
model.push_to_hub("MitchMooney/qwen3vl-8b-para-lora",
                  token = userdata.get("HF_TOKEN"), private = True)

## Deployment (after a good full run)
The model just replaces the 30B as the raw extractor; the Result Sink is unchanged.
Two paths:
- **MLX local inference** — MLX runs Qwen3-VL forward fine (only training was blocked).
  Merge the adapter, convert to MLX, add an mlx-vlm branch to `src/pdf_extract.py`
  (env-selected). Fully local.
- **GGUF + Ollama** — merge → convert to GGUF *plus the mmproj* → `ollama create`, then
  point `PARA_VL_MODEL` at it.
